# G-computation with link functions in PyMC

Gaussian {term}`Interrupted Time Series<Interrupted time series design>` models make the linear predictor, conditional expectation, and outcome scale look interchangeable. For generalized linear models they are not. CausalPy's default Bayesian impact subtracts a counterfactual **expected outcome** from the observed response (see {doc}`../knowledgebase/prediction-contract`). This notebook shows why link-scale contrasts fail, how to apply inverse links **draw by draw**, and how g-computation recovers response-scale causal effects from simulated Poisson and Bernoulli data.

:::{important}
**Empirical estimand (this notebook):** the average difference in expected outcomes between treated and control conditions on the **response scale** (counts for Poisson, probabilities for Bernoulli). Risk ratios, odds ratios, and link-scale regression coefficients are different estimands and are not the default reporting scale in CausalPy.
:::

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import seaborn as sns

FIG_WIDTH = 7
FIG_HEIGHT = 4
COLOR_TREATED = "#d62728"
COLOR_CONTROL = "#1f77b4"
COLOR_TRUE = "black"

sample_kwargs = {
    "chains": 2,
    "draws": 300,
    "tune": 300,
    "target_accept": 0.9,
    "random_seed": 42,
    "progressbar": False,
}
rng = np.random.default_rng(42)

## Poisson counts: the wrong and right contrasts

We simulate $n=200$ units with a binary treatment. The data-generating log-rate is $\eta_i = \beta_0 + \beta_1 T_i$ with $\beta_1 = 0.4$, so the true response-scale effect is $\mathbb{E}[Y\mid T{=}1] - \mathbb{E}[Y\mid T{=}0] = e^{\beta_0+\beta_1} - e^{\beta_0}$.

In [ ]:
n = 200
treatment = rng.binomial(1, 0.5, size=n)
beta0, beta1 = np.log(8.0), 0.4
eta = beta0 + beta1 * treatment
mu_true = np.exp(eta)
y = rng.poisson(mu_true)

true_ate = np.exp(beta0 + beta1) - np.exp(beta0)
df_poisson = pd.DataFrame({"treatment": treatment, "y": y})
df_poisson.head()

In [ ]:
with pm.Model() as poisson_model:
    X = pm.Data("X", df_poisson[["treatment"]].values)
    y_obs = pm.Data("y_obs", df_poisson["y"].values)
    beta = pm.Normal("beta", mu=0, sigma=1, shape=2)
    eta_p = beta[0] + beta[1] * X[:, 0]
    mu_p = pm.Deterministic("mu", pm.math.exp(eta_p))
    pm.Poisson("y_hat", mu=mu_p, observed=y_obs)
    idata_p = pm.sample(**sample_kwargs)

beta_draws = idata_p.posterior["beta"].stack(sample=("chain", "draw")).values
eta_treated = beta_draws[0] + beta_draws[1]
eta_control = beta_draws[0]
mu_treated = np.exp(eta_treated)
mu_control = np.exp(eta_control)

ate_gcomp = mu_treated - mu_control
ate_wrong_link = eta_treated - eta_control
ate_wrong_avg = np.exp(eta_treated.mean()) - np.exp(eta_control.mean())

summary = pd.DataFrame(
    {
        "estimand": [
            "True simulation ATE",
            "G-computation (draw-wise exp then contrast)",
            "Wrong: link-scale contrast",
            "Wrong: contrast of link-scale means",
        ],
        "mean": [
            true_ate,
            ate_gcomp.mean(),
            ate_wrong_link.mean(),
            ate_wrong_avg,
        ],
    }
)
summary

:::{warning}
Subtracting link-scale predictors from observed counts (what happens if `mu` stores $\eta$ instead of $e^\eta$) mixes log-rates and counts. Averaging $\eta$ before exponentiating is also biased by Jensen's inequality: $\exp(\mathbb{E}[\eta]) \neq \mathbb{E}[\exp(\eta)]$.
:::

The table shows only the draw-wise g-computation path recovers the simulated count-scale ATE within Monte Carlo error.

In [ ]:
fig, ax = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT))
sns.kdeplot(
    ate_gcomp,
    fill=True,
    alpha=0.35,
    color=COLOR_TREATED,
    label="G-computation ATE",
    ax=ax,
)
sns.kdeplot(
    ate_wrong_link,
    fill=True,
    alpha=0.25,
    color=COLOR_CONTROL,
    label="Link-scale contrast",
    ax=ax,
)
ax.axvline(true_ate, color=COLOR_TRUE, linestyle="--", linewidth=1.5, label="True ATE")
ax.set_xlabel("Average treatment effect (expected counts)")
ax.set_ylabel("Posterior density")
ax.legend(frameon=False)
fig.tight_layout()
fig

**Figure:** Posterior distribution of the Poisson ATE on the count scale. The g-computation estimator (inverse link applied per draw, then contrasted) centers on the simulated truth; the link-scale contrast is shifted because it lives on the log-rate scale.

## Bernoulli outcomes: probabilities, not logits

The same logic applies to logit models. The response-scale estimand is a difference in **probabilities** $\mathbb{E}[Y\mid T{=}1] - \mathbb{E}[Y\mid T{=}0]$, not a difference in logits.

In [ ]:
treatment_b = rng.binomial(1, 0.5, size=n)
b0, b1 = -0.2, 0.8
logit_p = b0 + b1 * treatment_b
p_true = 1 / (1 + np.exp(-logit_p))
y_b = rng.binomial(1, p_true)
true_ate_b = (1 / (1 + np.exp(-(b0 + b1)))) - (1 / (1 + np.exp(-b0)))

with pm.Model() as logit_model:
    Xb = pm.Data("Xb", treatment_b)
    yb = pm.Data("yb", y_b)
    gamma = pm.Normal("gamma", mu=0, sigma=1, shape=2)
    eta_b = gamma[0] + gamma[1] * Xb
    mu_b = pm.Deterministic("mu", pm.math.sigmoid(eta_b))
    pm.Bernoulli("y_hat", p=mu_b, observed=yb)
    idata_b = pm.sample(**sample_kwargs)

gamma_draws = idata_b.posterior["gamma"].stack(sample=("chain", "draw")).values
p_treated = 1 / (1 + np.exp(-(gamma_draws[0] + gamma_draws[1])))
p_control = 1 / (1 + np.exp(-gamma_draws[0]))
ate_b = p_treated - p_control
ate_b_wrong = (gamma_draws[0] + gamma_draws[1]) - gamma_draws[0]

pd.DataFrame(
    {
        "estimand": ["True simulation ATE", "G-computation", "Wrong: logit contrast"],
        "mean": [true_ate_b, ate_b.mean(), ate_b_wrong.mean()],
    }
)

A treatment coefficient in a nonlinear model is **not** generally the ATE unless the link is identity and there are no interactions. G-computation averages unit-level potential outcomes over the target population explicitly.

## Expected outcome vs posterior predictive noise

Causal impact in CausalPy uses `mu` (conditional expectation, no observation noise). Posterior predictive draws `y_hat` add sampling noise on top. For the Poisson model above, compare the width of $\mu$ and $y$ for a single treated unit:

In [ ]:
with poisson_model:
    ppc = pm.sample_posterior_predictive(
        idata_p, var_names=["mu", "y_hat"], progressbar=False
    )

mu_draws = ppc.posterior_predictive["mu"].values.flatten()
yhat_draws = ppc.posterior_predictive["y_hat"].values.flatten()
pd.Series({"sd(mu)": mu_draws.std(), "sd(y_hat)": yhat_draws.std()})

`sd(y_hat)` is larger because it includes Poisson sampling variability. Impact plots and default summaries focus on systematic causal effects via `mu`.

## Provider backends (`pymc-forecast`)

The optional {class}`~causalpy.pymc_forecast_models.PyMCForecastModel` adapter passes upstream `mu` / `mu_future` latents through to CausalPy impact calculation. Until `pymc-forecast` exposes a dedicated expected-observation output ([upstream #52](https://github.com/pymc-labs/pymc-forecast/issues/52)), linked GLM forecasts must supply the **inverse-linked expectation** as the latent so CausalPy subtracts quantities in compatible units. See {doc}`../knowledgebase/prediction-contract` for the full backend table.

## Summary

- Define the response-scale estimand before fitting.
- Register inverse-linked expectations as `mu` in custom PyMC models.
- Transform link-scale draws with the inverse link **before** contrasts and averaging.
- Fast regression coverage lives in `causalpy/tests/test_prediction_contract.py`.

:::{bibliography}
:filter: docname in docnames
:::